In [29]:
from google import genai
import pandas as pd
import time
import os

train_dataset = pd.read_csv("train_dataset.csv")
train_dataset['generated'] = ""


In [40]:
import traceback
batch_size = 10
start_idx, end_idx = 0, (len(train_dataset) // 2 - batch_size)
start_idx, end_idx = len(train_dataset) // 2, len(train_dataset)

# Add your own API key here
client = genai.Client(api_key="XXX")

with open("results.txt", "a") as f:
    pass

i = start_idx
while i < end_idx:
    try:
        batched_prompts = []
        for j in range(batch_size):
            if i + j >= end_idx:
                break
            row = train_dataset.loc[i + j]
            prompt = row['prompt']
            
            # Don't include the report
            prompt = prompt.split("### Poročilo")[0]
            batched_prompts.append(prompt)

        # Generate content for the batch of prompts
        prompt = "Generiraj 10 poročil o prometu na osnovi spodnjih podatkov. Vhodni podatki za vsako poročilo so ločeni z '### Vhodni podatki'. Odgovor naj vsebuje samo poročila.\n\n"
        prompt += "\n\n".join(batched_prompts)
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt
        )
        
        print("Processed index " + str(i))
        i+= batch_size
        
        with open("results.txt", "a") as f:
            f.write(f"Index {i}: {response.text}\n\n")
        
        # Sleep to avoid hitting API limits
        time.sleep(3)
            
    except Exception:
        print(traceback.format_exc())


Processed index 0
